le nom:
El jattioui 
le prenom: Maryame 
Master:GLCC

In [2]:
import numpy as np
from collections import Counter

# DATASET : On simule des données de Master
# X = [Moyenne Licence, Score Entretien Oral]
X_train = np.array([
    [15.5, 16], [12.0, 14], [14.0, 15], [10.5, 10], [11.0, 12], 
    [16.0, 18], [9.0, 8], [13.5, 13], [11.5, 11], [8.5, 9]
])

# y = [1: Admis, 0: Recalé]
y_train = np.array([1, 1, 1, 0, 0, 1, 0, 1, 0, 0])
class Node:
    """ 
    Structure de donnée pour représenter un élément de l'arbre.
    Un nœud est soit une branche (question), soit une feuille (réponse).
    """
    def __init__(self, feature=None, threshold=None, left=None, right=None, value=None):
        self.feature = feature     # Index de la caractéristique (ex: Moyenne)
        self.threshold = threshold # Valeur de coupure (ex: 12.0)
        self.left = left           # Enfant gauche (branche <= seuil)
        self.right = right         # Enfant droit (branche > seuil)
        self.value = value         # Si défini, c'est une feuille (résultat final)

class DecisionTreeMaster:
    def __init__(self, method='CART', max_depth=10, min_samples_split=2):
        """
        :param method: 'ID3', 'C4.5' ou 'CART'. Définit la logique mathématique.
        :param max_depth: Profondeur max pour éviter le sur-apprentissage (overfitting).
        :param min_samples_split: Nb minimum d'échantillons requis pour tenter une division.
        """
        self.method = method
        self.max_depth = max_depth
        self.min_samples_split = min_samples_split
        self.root = None # Racine de l'arbre, sera définie après fit()

    # --- MÉTRIQUES MATHÉMATIQUES ---

    def _entropy(self, y):
        """ Calcule le désordre (Shannon). Utilisé par ID3 et C4.5. """
        probas = np.bincount(y) / len(y)
        # Formule : - somme(p * log2(p)). Plus c'est haut, plus c'est désordonné.
        return -np.sum([p * np.log2(p) for p in probas if p > 0])

    def _gini(self, y):
        """ Calcule l'impureté de Gini. Utilisé par CART. """
        probas = np.bincount(y) / len(y)
        # Formule : 1 - somme(p^2). Indique la probabilité de mal classer un élément.
        return 1 - np.sum([p**2 for p in probas])

    # --- LOGIQUE DE SÉLECTION DU MEILLEUR SPLIT ---

    def _calculate_gain(self, y, left_y, right_y):
        """ Calcule la qualité de la division selon l'algorithme choisi. """
        n = len(y)
        n_l, n_r = len(left_y), len(right_y)

        if self.method == 'CART':
            # --- LOGIQUE CART ---
            parent_loss = self._gini(y)
            child_loss = (n_l/n)*self._gini(left_y) + (n_r/n)*self._gini(right_y)
            return parent_loss - child_loss # Réduction d'impureté
        
        else:
            # --- LOGIQUE ID3 / C4.5 ---
            parent_loss = self._entropy(y)
            child_loss = (n_l/n)*self._entropy(left_y) + (n_r/n)*self._entropy(right_y)
            gain = parent_loss - child_loss # Gain d'information
            
            if self.method == 'C4.5':
                # Normalisation : Gain Ratio pour éviter le biais des colonnes à valeurs multiples
                split_info = self._entropy([0]*n_l + [1]*n_r)
                return gain / split_info if split_info > 0 else 0
            
            return gain # Simple gain pour ID3

    # --- CONSTRUCTION RÉCURSIVE DE L'ARBRE ---

    def fit(self, X, y):
        """ Démarre la construction à partir de la racine. """
        self.root = self._grow_tree(X, y)

    def _grow_tree(self, X, y, depth=0):
        """ Fonction récursive qui divise les données en sous-groupes. """
        n_samples, n_features = X.shape
        n_labels = len(np.unique(y))

        # 1. VÉRIFIER LES CONDITIONS D'ARRÊT (C'est la base de la récursivité)
        if (depth >= self.max_depth or n_labels == 1 or n_samples < self.min_samples_split):
            # On crée une feuille : on prend la classe la plus fréquente (Majorité)
            leaf_value = Counter(y).most_common(1)[0][0]
            return Node(value=leaf_value)

        # 2. TROUVER LA MEILLEURE QUESTION (FEATURE + SEUIL)
        best_feat, best_thresh = self._best_split(X, y)

        # 3. DIVISER LES DONNÉES ET PASSER AUX ENFANTS
        # On sépare les indices selon le seuil trouvé
        left_idxs = np.argwhere(X[:, best_feat] <= best_thresh).flatten()
        right_idxs = np.argwhere(X[:, best_feat] > best_thresh).flatten()

        # Appel récursif pour construire les branches gauche et droite
        left = self._grow_tree(X[left_idxs, :], y[left_idxs], depth + 1)
        right = self._grow_tree(X[right_idxs, :], y[right_idxs], depth + 1)
        
        return Node(feature=best_feat, threshold=best_thresh, left=left, right=right)

    def _best_split(self, X, y):
        """ Parcourt chaque colonne et chaque valeur pour trouver le meilleur gain. """
        best_gain = -1
        split_idx, split_thresh = 0, 0

        for feat_idx in range(X.shape[1]): # On teste chaque colonne
            column = X[:, feat_idx]
            thresholds = np.unique(column) # On teste chaque valeur unique comme seuil possible
            
            for thr in thresholds:
                left_y = y[column <= thr]
                right_y = y[column > thr]
                
                if len(left_y) == 0 or len(right_y) == 0:
                    continue # On ignore les divisions vides
                
                gain = self._calculate_gain(y, left_y, right_y)
                
                if gain > best_gain: # On garde le meilleur résultat
                    best_gain = gain
                    split_idx = feat_idx
                    split_thresh = thr
        return split_idx, split_thresh

    # --- UTILISATION DU MODÈLE (PRÉDICTION) ---

    def predict(self, X):
        """ Reçoit un tableau de données et retourne les prédictions. """
        return np.array([self._traverse_tree(x, self.root) for x in X])

    def _traverse_tree(self, x, node):
        """ Descend dans l'arbre pour un échantillon x jusqu'à une feuille. """
        if node.value is not None: # Si on est sur une feuille, on a la réponse !
            return node.value
        
        # Sinon, on compare la valeur de x au seuil du nœud actuel
        if x[node.feature] <= node.threshold:
            return self._traverse_tree(x, node.left) # On va à gauche
        return self._traverse_tree(x, node.right)    # On va à droite
# =================================================================
# 1. ENTRAÎNEMENT DE ID3 (Information Gain pur)
# =================================================================
print(f"{'='*10} TEST ID3 {'='*10}")
# On initialise avec 'ID3' : utilise l'Entropie de Shannon
clf_id3 = DecisionTreeMaster(method='ID3', max_depth=5)
clf_id3.fit(X_train, y_train) # L'arbre se construit en maximisant le gain d'info
nouvel_etudiant = np.array([[14.5, 15]])
# Prédiction
pred_id3 = clf_id3.predict(nouvel_etudiant)
print(f"Décision ID3 : {'ADMIS' if pred_id3[0] == 1 else 'REJETÉ'}")


# =================================================================
# 2. ENTRAÎNEMENT DE C4.5 (Gain Ratio)
# =================================================================
print(f"\n{'='*10} TEST C4.5 {'='*10}")
# On initialise avec 'C4.5' : utilise le Gain Ratio (normalisation)
# Utile pour ne pas être biaisé par des variables ayant trop de valeurs uniques
clf_c45 = DecisionTreeMaster(method='C4.5', max_depth=5)
clf_c45.fit(X_train, y_train)
nouvel_etudiant = np.array([[14.5, 15]])
# Prédiction
pred_c45 = clf_c45.predict(nouvel_etudiant)
print(f"Décision C4.5 : {'ADMIS' if pred_c45[0] == 1 else 'REJETÉ'}")


# =================================================================
# 3. ENTRAÎNEMENT DE CART (Gini Impurity)
# =================================================================
print(f"\n{'='*10} TEST CART {'='*10}")
# On initialise avec 'CART' : utilise l'Indice de Gini
# C'est l'algorithme le plus rapide (utilisé par Scikit-Learn)
clf_cart = DecisionTreeMaster(method='CART', max_depth=5)
clf_cart.fit(X_train, y_train)

# Prédiction
pred_cart = clf_cart.predict(nouvel_etudiant)
print(f"Décision CART : {'ADMIS' if pred_cart[0] == 1 else 'REJETÉ'}")
    
  


========== TEST ID3 ==========


NameError: name 'nouvel_etudiant' is not defined